# Control de calidad de la normalización y corrección rápida (sin rehacer el preprocesamiento)

**Idea central.** El análisis de primer nivel de cada sujeto es estadísticamente válido aunque su normalización a MNI haya quedado desplazada o escalada: la *ubicación* en MNI es lo que falla, no el GLM. Por eso, en los sujetos con mala normalización **no hay que repetir el BOLD ni el GLM**: se estima una pequeña corrección residual (mean_mni → plantilla) y se aplica **solo a los mapas del primer nivel** (un volumen cada uno), lo que toma segundos por sujeto en lugar de horas.

1. **Parte A — medir (no a ojo).** Para los 46 sujetos: Dice con el cerebro MNI, razón de tamaño, desplazamiento del centro y extensión en x/y/z.
2. **Parte B — ver.** Montaje con el borde del cerebro MNI (verde) y el del sujeto (rojo) sobre la media funcional.
3. **Parte C — corregir.** Registro afín residual de la media funcional a la plantilla, aplicado a los mapas del GLM, con comprobación antes/después.

## 1. Rutas e importaciones

In [ ]:
import gc, shutil, warnings
from pathlib import Path

import ants
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img
from nilearn.masking import compute_epi_mask
from scipy.ndimage import binary_fill_holes, center_of_mass

RAIZ        = Path.cwd().resolve()                    # el notebook esta en Proyecto_U3
CSV_SUJETOS = RAIZ / "sujetos_46.csv"
OUT_DIR     = RAIZ / "derivatives_unidad3"            # preprocesados (mean_mni, transforms)
MNI_PATH    = Path(r"C:\fMRI\Worskshop-fMRI-preprocessing-basics\data\external\templates\MNI152_T1_2mm.nii.gz")
TEMPLATE_MASK = None      # opcional: ruta a una mascara cerebral MNI 2 mm (ej. MNI152_T1_2mm_brain_mask.nii.gz). None = mascara de nilearn

# Carpeta con los mapas del primer nivel: GLM_DIR / <sub> / archivos (AJUSTA GLM_DIR si tus carpetas estan en otra ruta)
GLM_DIR     = OUT_DIR / "glm_primer_nivel"
# Archivos a corregir y como interpolarlos (las mascaras con vecino mas cercano para que sigan siendo 0/1)
PATRONES_MAPAS = {
    "*contrast-move_stat-effect.nii.gz":   "linear",            # beta / tamano del efecto (el que se usa en el 2o nivel)
    "*contrast-move_stat-variance.nii.gz": "linear",
    "*contrast-move_stat-z.nii.gz":        "linear",
    "*desc-fdr05_statmap.nii.gz":          "linear",            # mapa Z umbralizado FDR (solo visualizacion)
    "*desc-brain_mask.nii.gz":             "nearestNeighbor",   # mascara cerebral del sujeto
}
CORR_DIR    = OUT_DIR / "glm_corregido_mni"           # salida corregida
QC_DIR      = OUT_DIR / "qc_alineacion"
for d in (CORR_DIR, QC_DIR): d.mkdir(parents=True, exist_ok=True)

for n, p in [("CSV", CSV_SUJETOS), ("OUT", OUT_DIR), ("MNI", MNI_PATH), ("GLM", GLM_DIR)]:
    print(f"{n:4s} {'OK ' if p.exists() else 'NO EXISTE'} {p}")
df = pd.read_csv(CSV_SUJETOS)

## Parte A — Métricas objetivas de la normalización

In [ ]:
def mascara_referencia():
    """Cerebro MNI en la misma rejilla que la plantilla del proyecto."""
    tpl = nib.load(str(MNI_PATH))
    if TEMPLATE_MASK:
        m = nib.load(str(TEMPLATE_MASK))
    else:
        from nilearn.datasets import load_mni152_brain_mask
        m = load_mni152_brain_mask(resolution=2)          # viene incluida en nilearn, sin descarga
    m = resample_to_img(m, tpl, interpolation="nearest", force_resample=True, copy_header=True)
    return m.get_fdata() > 0.5, tpl.affine

def mascara_sujeto(mean_path):
    m = compute_epi_mask(str(mean_path)).get_fdata() > 0.5
    return binary_fill_holes(m)

def extension_mm(mask, affine):
    """Largo (mm) del cerebro en x, y, z usando percentiles 1-99 para ignorar vóxeles sueltos."""
    ijk = np.argwhere(mask)
    xyz = nib.affines.apply_affine(affine, ijk)
    lo, hi = np.percentile(xyz, 1, axis=0), np.percentile(xyz, 99, axis=0)
    return hi - lo

def metricas(mean_path, ref_mask, affine):
    s = mascara_sujeto(mean_path)
    inter = (s & ref_mask).sum()
    dice = 2 * inter / (s.sum() + ref_mask.sum())
    vol_ratio = s.sum() / ref_mask.sum()
    cs = nib.affines.apply_affine(affine, center_of_mass(s))
    cr = nib.affines.apply_affine(affine, center_of_mass(ref_mask))
    ext_s, ext_r = extension_mm(s, affine), extension_mm(ref_mask, affine)
    return dict(dice=dice, razon_volumen=vol_ratio,
                desplaz_centro_mm=float(np.linalg.norm(cs - cr)),
                dx_mm=float(cs[0] - cr[0]), dy_mm=float(cs[1] - cr[1]), dz_mm=float(cs[2] - cr[2]),
                razon_ext_x=ext_s[0] / ext_r[0], razon_ext_y=ext_s[1] / ext_r[1], razon_ext_z=ext_s[2] / ext_r[2])

def guardar_fig(fig, nombre, dpi=110):
    """Guarda la figura con un nombre limpio; si falla, avisa y muestra la ruta exacta (sin romper el notebook)."""
    import re
    limpio = re.sub(r"[^A-Za-z0-9_-]+", "_", str(nombre)).strip("_") or "figura"
    ruta = QC_DIR / (limpio + ".png")
    try:
        fig.savefig(ruta, dpi=dpi)
        print("Guardado:", ruta)
    except OSError as e:
        print("No se pudo guardar el PNG:", repr(e))
        print("Ruta intentada (repr):", repr(str(ruta)))
    return ruta

def ruta_mean(sub):
    r = df[df["sub"] == sub].iloc[0]
    return OUT_DIR / sub / f"{sub}_ses-{r['ses']}_task-Motor2Class_run-{int(r['run'])}_mean_mni.nii.gz"

In [ ]:
ref_mask, aff_ref = mascara_referencia()
filas = []
for _, r in df.iterrows():
    mp = ruta_mean(r["sub"])
    if not mp.exists():
        print("sin mean_mni:", r["sub"]); continue
    filas.append(dict(sub=r["sub"], grupo=r["grupo"], age=r["age"], TR=r["TR"], **metricas(mp, ref_mask, aff_ref)))
qc = pd.DataFrame(filas)

def mad_z(x):
    med = np.median(x); mad = 1.4826 * np.median(np.abs(x - med))
    return (x - med) / (mad if mad > 0 else 1)

# Marca como atipicos SOLO a los peores que el resto (z robusto negativo) o con valores absolutos malos
qc["z_dice"] = mad_z(qc["dice"]); qc["z_vol"] = mad_z(qc["razon_volumen"])
qc["marcado"] = ((qc["z_dice"] < -2.5) | (qc["z_vol"] < -2.5) | (qc["dice"] < 0.70) |
                 (qc["desplaz_centro_mm"] > 8) | (qc[["razon_ext_x", "razon_ext_y"]].min(axis=1) < 0.88))
qc.to_csv(QC_DIR / "qc_alineacion_46.csv", index=False)
print(f"Marcados: {int(qc['marcado'].sum())} de {len(qc)}")
qc.sort_values("dice").round(2)[["sub", "grupo", "age", "TR", "dice", "razon_volumen", "desplaz_centro_mm", "razon_ext_x", "razon_ext_y", "razon_ext_z", "marcado"]].head(15)

In [ ]:
# ¿Los problemas se relacionan con la edad (ninos/adolescentes) o con el protocolo (TR)?
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for g, col in [("sano", "tab:blue"), ("epilepsia", "tab:red")]:
    q = qc[qc.grupo == g]
    ax[0].scatter(q["age"], q["razon_volumen"], c=col, label=g, s=28)
    ax[1].scatter(q["age"], q["dice"], c=col, label=g, s=28)
for a, t in zip(ax, ["Razon de volumen del cerebro (sujeto / MNI)", "Dice con el cerebro MNI"]):
    a.set_xlabel("edad (anios)"); a.set_title(t); a.axvline(18, color="gray", ls="--", lw=0.8)
ax[0].axhline(1, color="k", lw=0.6); ax[0].legend()
fig.tight_layout(); fig.savefig(QC_DIR / "metricas_vs_edad.png", dpi=120); plt.show()
print(qc.groupby(qc["age"] < 18)[["dice", "razon_volumen"]].agg(["mean", "min"]).round(2))

## Parte B — Montaje visual con bordes del cerebro
**Verde** = borde del cerebro MNI (la referencia). **Rojo** = borde del cerebro del sujeto. Si el rojo queda **dentro** del verde, el cerebro quedó pequeño; si queda desplazado hacia un lado o inclinado, hay un error de posición/rotación. (El contorno anterior usaba un percentil de la plantilla, que dibuja estructuras internas y no el borde del cerebro, por eso era difícil juzgar el tamaño.)

In [ ]:
def montaje(subs, nombre="montaje", filas_por_figura=6):
    tpl = nib.load(str(MNI_PATH))
    cz = [int(round(v)) for v in center_of_mass(ref_mask)]
    for k in range(0, len(subs), filas_por_figura):
        grupo = subs[k:k + filas_por_figura]
        fig, axes = plt.subplots(len(grupo), 3, figsize=(9, 2.7 * len(grupo)), squeeze=False)
        for i, s in enumerate(grupo):
            mp = ruta_mean(s)
            mean = nib.load(str(mp)).get_fdata(); sm = mascara_sujeto(mp)
            vmax = np.percentile(mean, 99)
            for j, (nom, eje) in enumerate([("Sagital", 0), ("Coronal", 1), ("Axial", 2)]):
                sl = lambda a: np.take(a, cz[eje], axis=eje).T
                ax = axes[i, j]
                ax.imshow(sl(mean), cmap="gray", origin="lower", vmin=0, vmax=vmax)
                ax.contour(sl(ref_mask.astype(float)), levels=[0.5], colors="lime", linewidths=1.0)
                ax.contour(sl(sm.astype(float)), levels=[0.5], colors="red", linewidths=1.0)
                ax.set_title(f"{s} - {nom}" if j == 0 else nom, fontsize=9); ax.axis("off")
        fig.tight_layout(); guardar_fig(fig, f"{nombre}_{k // filas_por_figura + 1}"); plt.show()

marcados = list(qc.loc[qc["marcado"], "sub"])
print("Marcados:", marcados)
montaje(marcados if marcados else list(qc.sort_values("dice")["sub"].head(6)), "marcados")

Revisa también unos pocos sujetos **no** marcados para comprobar que el criterio es razonable: `montaje(["sub-065", "sub-076"], "control")`.
Si con tu ojo hay sujetos mal normalizados que no salieron marcados, añádelos a mano en la lista de la Parte C.

## Parte C — Corrección residual (afín) aplicada a los mapas del primer nivel
Se registra la **media funcional en MNI** del sujeto contra la plantilla con una transformación afín (traslación, rotación, **escala** y cizalla), que es lo que corrige un cerebro pequeño o inclinado, y esa transformación se aplica a los mapas del GLM. No se toca el BOLD ni se repite el GLM.
- Se aplica a los mapas de **efecto (β)**, varianza, Z, mapa FDR y la máscara cerebral de cada sujeto (la máscara con vecino más cercano). Para el segundo nivel usa sobre todo el efecto y la máscara.
- Se comprueba con las mismas métricas **antes y después**; solo se conservan las correcciones que mejoran.

In [ ]:
TIPO_RESIDUAL = "Affine"      # "Affine" corrige tamano/inclinacion global. "SyN" tambien deforma localmente (riesgo de sobreajuste con imagen borrosa)

def corregir_sujeto(sub):
    mp = ruta_mean(sub)
    d_out = CORR_DIR / sub; d_out.mkdir(parents=True, exist_ok=True)
    tpl = ants.image_read(str(MNI_PATH))
    mean = ants.image_read(str(mp))
    reg = ants.registration(fixed=tpl, moving=mean, type_of_transform=TIPO_RESIDUAL, aff_metric="mattes")
    tx = []
    for i, f in enumerate(reg["fwdtransforms"]):
        f = Path(f); dst = d_out / f"residual_{i}{''.join(f.suffixes)}"
        shutil.copy(f, dst); tx.append(str(dst))

    # media corregida (para medir antes/despues)
    mean_corr = ants.apply_transforms(fixed=tpl, moving=mean, transformlist=tx, interpolator="linear")
    mean_corr_p = d_out / f"{sub}_mean_mni_realign.nii.gz"; ants.image_write(mean_corr, str(mean_corr_p))

    # mapas del GLM
    mapas = []
    for pat, interp in PATRONES_MAPAS.items():
        for p in sorted((GLM_DIR / sub).glob(pat)):
            if "realign" in p.name:
                continue
            m = ants.image_read(str(p))
            w = ants.apply_transforms(fixed=tpl, moving=m, transformlist=tx, interpolator=interp)
            ants.image_write(w, str(d_out / (p.name.replace(".nii.gz", "") + "_realign.nii.gz")))
            mapas.append(p.name)
    if not mapas:
        print(f"   [aviso] {sub}: no se encontraron mapas en {GLM_DIR / sub}")
    antes = metricas(mp, ref_mask, aff_ref)
    despues = metricas(mean_corr_p, ref_mask, aff_ref)
    del tpl, mean, mean_corr, reg; gc.collect()
    return dict(sub=sub, n_mapas=len(mapas),
                dice_antes=antes["dice"], dice_despues=despues["dice"],
                vol_antes=antes["razon_volumen"], vol_despues=despues["razon_volumen"],
                centro_antes_mm=antes["desplaz_centro_mm"], centro_despues_mm=despues["desplaz_centro_mm"])

In [ ]:
# Sujetos a corregir: los marcados + los que quieras anadir a mano
EXTRA = []                                  # ej. ["sub-065"]
CORREGIR_TODOS = False                      # True = aplica la correccion a los 46 y decide despues con la Parte D
SUJETOS_A_CORREGIR = list(df["sub"]) if CORREGIR_TODOS else sorted(set(marcados) | set(EXTRA))
print("Se corregiran:", SUJETOS_A_CORREGIR)

res = []
for s in SUJETOS_A_CORREGIR:
    try:
        r = corregir_sujeto(s); r["mejora"] = r["dice_despues"] > r["dice_antes"] + 0.01
        res.append(r); print(f"{s}: Dice {r['dice_antes']:.2f} -> {r['dice_despues']:.2f} | volumen {r['vol_antes']:.2f} -> {r['vol_despues']:.2f}")
    except Exception as e:
        print(f"{s}: ERROR {e!r}")
corr = pd.DataFrame(res)
corr.to_csv(QC_DIR / "resultado_correccion.csv", index=False)
corr.round(2)

In [ ]:
# Comprobacion visual de lo corregido: rojo = cerebro del sujeto tras la correccion
def montaje_corregidos(subs):
    tpl = nib.load(str(MNI_PATH)); cz = [int(round(v)) for v in center_of_mass(ref_mask)]
    fig, axes = plt.subplots(len(subs), 3, figsize=(9, 2.7 * len(subs)), squeeze=False)
    for i, s in enumerate(subs):
        mp = CORR_DIR / s / f"{s}_mean_mni_realign.nii.gz"
        mean = nib.load(str(mp)).get_fdata(); sm = mascara_sujeto(mp); vmax = np.percentile(mean, 99)
        for j, (nom, eje) in enumerate([("Sagital", 0), ("Coronal", 1), ("Axial", 2)]):
            sl = lambda a: np.take(a, cz[eje], axis=eje).T
            ax = axes[i, j]; ax.imshow(sl(mean), cmap="gray", origin="lower", vmin=0, vmax=vmax)
            ax.contour(sl(ref_mask.astype(float)), levels=[0.5], colors="lime", linewidths=1.0)
            ax.contour(sl(sm.astype(float)), levels=[0.5], colors="red", linewidths=1.0)
            ax.set_title(f"{s} corregido - {nom}" if j == 0 else nom, fontsize=9); ax.axis("off")
    fig.tight_layout(); guardar_fig(fig, "montaje_corregidos"); plt.show()

if len(corr):
    montaje_corregidos(list(corr["sub"].head(6)))

## Parte D — Validación con un punto de referencia funcional (la mano en M1)
El Dice **no basta** para decidir si una corrección es buena: mide solo la forma de la máscara. Si el campo de visión del BOLD no cubre la parte inferior del cerebro (cerebelo y tronco), un registro afín puede **agrandar** el cerebro para que llene la plantilla y distorsionar la anatomía. Un criterio más sólido: **¿cae la activación de la mano derecha donde debe?**
Se busca el pico del mapa Z dentro de una caja alrededor de la corteza motora izquierda (`CAJA`) y se mide su distancia a una coordenada de referencia aproximada (`REF`, la zona de la mano). Se compara **antes y después** de la corrección: si la distancia baja y la dispersión entre sujetos disminuye, la corrección mejora la alineación anatómica.
Advertencia: la distancia absoluta a `REF` varía entre personas (unos 6-8 mm es normal); lo importante es el **cambio** y la **dispersión** del grupo.

In [ ]:
CAJA = dict(x=(-60, -15), y=(-45, -5), z=(40, 72))     # corteza motora izquierda en MNI (mm)
REF  = np.array([-38.0, -22.0, 56.0])                  # zona de la mano en M1 (aproximada)

def pico_motor(z_path):
    img = nib.load(str(z_path)); z = img.get_fdata()
    ijk = np.indices(z.shape).reshape(3, -1).T
    xyz = nib.affines.apply_affine(img.affine, ijk).reshape(z.shape + (3,))
    dentro = ((xyz[..., 0] >= CAJA["x"][0]) & (xyz[..., 0] <= CAJA["x"][1]) &
              (xyz[..., 1] >= CAJA["y"][0]) & (xyz[..., 1] <= CAJA["y"][1]) &
              (xyz[..., 2] >= CAJA["z"][0]) & (xyz[..., 2] <= CAJA["z"][1]))
    zz = np.where(dentro & np.isfinite(z), z, -np.inf)
    k = np.unravel_index(np.argmax(zz), z.shape)
    return xyz[k], float(zz[k])

def archivo_z(sub, corregido=False):
    base = (CORR_DIR if corregido else GLM_DIR) / sub
    pat = "*contrast-move_stat-z_realign.nii.gz" if corregido else "*contrast-move_stat-z.nii.gz"
    f = sorted(base.glob(pat))
    return f[0] if f else None

filas = []
for s in df["sub"]:
    f0 = archivo_z(s)
    if f0 is None:
        continue
    p0, z0 = pico_motor(f0)
    fila = dict(sub=s, x0=p0[0], y0=p0[1], z0=p0[2], zmax0=z0, dist0=np.linalg.norm(p0 - REF))
    f1 = archivo_z(s, corregido=True)
    if f1 is not None:
        p1, z1 = pico_motor(f1)
        fila.update(x1=p1[0], y1=p1[1], z1=p1[2], zmax1=z1, dist1=np.linalg.norm(p1 - REF))
    filas.append(fila)
pk = pd.DataFrame(filas)
pk["corregido"] = pk["dist1"].notna() if "dist1" in pk else False
if "dist1" in pk:
    pk["acepta_correccion"] = pk["corregido"] & (pk["dist1"] < pk["dist0"] - 1.0)   # mejora de al menos 1 mm
pk.to_csv(QC_DIR / "validacion_pico_motor.csv", index=False)
pk.round(1)

In [ ]:
# Dispersion del pico motor entre sujetos: original vs. "mixto" (corregido donde se acepto la correccion)
orig = pk[["x0", "y0", "z0"]].to_numpy()
mixto = orig.copy()
if "acepta_correccion" in pk:
    for i, r in pk.iterrows():
        if r.get("acepta_correccion", False):
            mixto[i] = [r["x1"], r["y1"], r["z1"]]
sd = lambda a: np.sqrt(((a - a.mean(axis=0)) ** 2).sum(axis=1).mean())
print(f"Dispersion del pico (distancia media al centro del grupo): original {sd(orig):.1f} mm | con correcciones aceptadas {sd(mixto):.1f} mm")
if "dist1" in pk:
    print("Corregidos:", int(pk["corregido"].sum()), "| aceptados:", int(pk.get("acepta_correccion", pd.Series([0])).sum()))
    print(pk.loc[pk["corregido"], ["sub", "dist0", "dist1", "acepta_correccion"]].round(1).to_string(index=False))

## Parte E — Cobertura del campo de visión (¿hasta dónde llega el BOLD de cada sujeto?)
En los montajes se ve que en varios sujetos la imagen funcional **termina abajo con un corte plano**: el BOLD no cubre cerebelo y tronco. Aquí se mide con la máscara cerebral que ya guardó el primer nivel (`*desc-brain_mask.nii.gz`): el límite inferior cubierto por cada sujeto (en mm, coordenada z de MNI) y un mapa de **cobertura** (en qué fracción de los sujetos hay señal en cada vóxel). Con eso se define la **máscara de grupo** para el segundo nivel: solo se analizan los vóxeles cubiertos por casi todos.

In [ ]:
COBERTURA_MIN = 0.90      # un voxel entra a la mascara de grupo si lo cubre al menos este % de sujetos

def archivo_mascara(sub):
    f = sorted((GLM_DIR / sub).glob("*desc-brain_mask.nii.gz"))
    return f[0] if f else None

filas, suma, n, ref_img = [], None, 0, None
suma_g, n_g = {}, {}
for _, r in df.iterrows():
    f = archivo_mascara(r["sub"])
    if f is None:
        continue
    img = nib.load(str(f)); m = img.get_fdata() > 0.5
    z = nib.affines.apply_affine(img.affine, np.argwhere(m))[:, 2]
    filas.append(dict(sub=r["sub"], grupo=r["grupo"], limite_inferior_mm=np.percentile(z, 1),
                      limite_superior_mm=np.percentile(z, 99), n_voxeles=int(m.sum())))
    suma = m.astype(np.uint16) if suma is None else suma + m
    suma_g[r["grupo"]] = suma_g.get(r["grupo"], 0) + m.astype(np.uint16)
    n_g[r["grupo"]] = n_g.get(r["grupo"], 0) + 1
    n += 1; ref_img = img
cob = pd.DataFrame(filas)
cob.to_csv(QC_DIR / "cobertura_limite_inferior.csv", index=False)
print(cob.groupby("grupo")["limite_inferior_mm"].describe().round(1))
print("Sujetos con limite inferior mas alto (menos cobertura abajo):")
print(cob.sort_values("limite_inferior_mm", ascending=False).head(8).round(1).to_string(index=False))

frac = suma / n
mask_grupo = frac >= COBERTURA_MIN
# Por grupo: el voxel debe estar cubierto en CADA grupo (importante para la comparacion sanos vs epilepsia)
mask_por_grupo = np.all([(suma_g[g] / n_g[g]) >= COBERTURA_MIN for g in suma_g], axis=0)

def guardar_nifti(arr, nombre, dtype=np.uint8):
    img = nib.Nifti1Image(arr.astype(dtype), ref_img.affine)
    img.set_data_dtype(dtype)
    nib.save(img, str(QC_DIR / nombre))

guardar_nifti(frac, "cobertura_fraccion.nii.gz", np.float32)
guardar_nifti(mask_grupo, "mascara_grupo_cobertura.nii.gz")
guardar_nifti(mask_por_grupo, "mascara_grupo_cobertura_por_grupo.nii.gz")

# --- Comprobacion de la zona motora: SOLO voxeles que son cerebro MNI dentro de la caja ---
ijk = np.indices(mask_grupo.shape).reshape(3, -1).T
xyz = nib.affines.apply_affine(ref_img.affine, ijk).reshape(mask_grupo.shape + (3,))
caja = ((xyz[..., 0] >= CAJA["x"][0]) & (xyz[..., 0] <= CAJA["x"][1]) & (xyz[..., 1] >= CAJA["y"][0]) &
        (xyz[..., 1] <= CAJA["y"][1]) & (xyz[..., 2] >= CAJA["z"][0]) & (xyz[..., 2] <= CAJA["z"][1]))
caja_cerebro = caja & ref_mask          # la caja tambien incluye aire/craneo: no cuentan
print(f"Caja motora: {int(caja.sum())} voxeles, de los cuales {int(caja_cerebro.sum())} ({caja_cerebro.sum() / caja.sum():.1%}) son cerebro MNI")
print(f"Mascara de grupo (>= {COBERTURA_MIN:.0%}): cubre {mask_grupo[caja_cerebro].mean():.1%} del cerebro dentro de la caja motora | por grupo: {mask_por_grupo[caja_cerebro].mean():.1%}")

print("Sensibilidad al umbral de cobertura (cerebro dentro de la caja motora cubierto):")
for u in [0.7, 0.8, 0.9, 0.95, 1.0]:
    print(f"  >= {u:.0%} de sujetos: {(frac[caja_cerebro] >= u).mean():.1%} de la caja | {int((frac >= u).sum())} voxeles en total")

print("Perfil por altura (% del cerebro MNI cubierto por la mascara de grupo):")
for lo, hi in [(40, 90), (0, 40), (-20, 0), (-90, -20)]:
    sel = (xyz[..., 2] >= lo) & (xyz[..., 2] < hi) & ref_mask
    print(f"  z {lo:>4} a {hi:>4} mm: {mask_grupo[sel].mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for g, col in [("sano", "tab:blue"), ("epilepsia", "tab:red")]:
    ax[0].hist(cob.loc[cob.grupo == g, "limite_inferior_mm"], bins=12, alpha=0.6, color=col, label=g)
ax[0].set_xlabel("limite inferior cubierto (mm, z MNI)"); ax[0].set_ylabel("sujetos"); ax[0].legend()
ax[0].set_title("Hasta donde llega el BOLD por abajo")
k = mask_grupo.shape[0] // 2
ax[1].imshow(frac[k].T, origin="lower", cmap="viridis", vmin=0, vmax=1); ax[1].set_title("Fraccion de sujetos con cobertura (corte sagital medio)")
ax[1].axis("off")
fig.tight_layout(); fig.savefig(QC_DIR / "cobertura_fov.png", dpi=120); plt.show()

## Qué hacer con el resultado
- **Si el montaje muestra un corte plano abajo (campo de visión recortado):** la normalización de la parte alta del cerebro está bien y la corrección afín **no es necesaria**; de hecho estira el cerebro para llenar la plantilla. Usa los mapas **originales** y trabaja con la **máscara de cobertura** de la Parte E en el segundo nivel. Cerebelo y tronco no serán analizables de forma fiable.
- **Si la Parte D acepta la corrección** (el pico motor se acerca a la referencia y la dispersión baja): usa los mapas `*_realign.nii.gz` solo para esos sujetos.
- **Si hay sujetos claramente mal normalizados** (no solo recortados): para esos pocos hay que repetir el registro T1→MNI con ajustes y volver a aplicar las transformaciones.
- **Reporta en el informe** el campo de visión incompleto, qué se corrigió (si algo) y las métricas.